# Colombo RM Forecast-Space OR-Tools Demo

This notebook documents the manager-facing Colombo Main Warehouse raw-material planning demo. It reads the operational planning artifacts generated under `Ai miroservices/modeling/outputs/colombo_rm_operational_demo/` and explains the stock policy, confidence, and OR-Tools slotting proof points.

In [ ]:
from pathlib import Path
import json
import pandas as pd

OUT = Path('../outputs/colombo_rm_operational_demo').resolve()
forecast = pd.read_csv(OUT / 'forecast_results_colombo_rm.csv')
issues = pd.read_csv(OUT / 'material_issue_history_colombo_rm.csv')
supplier_rules = pd.read_csv(OUT / 'supplier_rules_colombo_rm.csv')
inventory = pd.read_csv(OUT / 'inventory_batches_colombo_rm.csv')
quality = json.loads((OUT / 'quality_report.json').read_text())
quality

## Stock Policy Formula

- Expected demand = sum P50 over the selected horizon.
- High-demand case = sum P90.
- Low-demand case = sum P10.
- Safety stock includes both demand variation and lead-time variation.
- Final order quantity is rounded by minimum order and order multiple.

In [ ]:
horizon = forecast.sort_values('forecast_period').groupby('material_code').head(6)
policy = horizon.groupby('material_code').agg(
    expected_demand=('forecast_p50', 'sum'),
    low_demand=('forecast_p10', 'sum'),
    high_demand=('forecast_p90', 'sum'),
).merge(supplier_rules, on='material_code', how='left')
policy['daily_expected_demand'] = policy['expected_demand'] / 180
policy['daily_sigma'] = (policy['high_demand'] - policy['low_demand']) / (2 * 1.2816 * 180)
policy['lead_time_demand'] = policy['daily_expected_demand'] * policy['lead_time_days_mean']
policy.head()

## OR-Tools Slotting Proof

The production service runs the assignment model inside `ai_services/slotting-service`. The Spring backend calls that microservice first and stores fallback metadata only if the service is unavailable or infeasible.

In [ ]:
try:
    from ortools.linear_solver import pywraplp
    solver_available = pywraplp.Solver.CreateSolver('SCIP') is not None or pywraplp.Solver.CreateSolver('CBC') is not None
except Exception:
    solver_available = False
solver_available

## Manager Confidence

The UI should present confidence as planning confidence, not a guarantee. High-confidence recommendations require good forecast coverage, supplier rules, location capacity, feasible solver status, and stable sensitivity.